# 📱 Krea 2 & 顶级日漫手绘工作站（手机远程一键懒人版）

**专为手机远程操作优化！无需上传任何文件，我在你的 Mac 本地准备好训练集后会自动同步到这里，你只需要用手指点一下对应的 `▶` 按钮即可！**

> **💡 唯一需要检查的设置**：点击右上角 `连接 (Connect)` 旁的小箭头 `▼` $\rightarrow$ `更改运行时类型` $\rightarrow$ 选 **`A100 GPU`**（练 LoRA 满血首选）或 **`L4 GPU`**（日常画图首选）。

In [ ]:
#@title ## 🎨 【按钮 1】一键启动 ComfyUI 漫画画图工作台（点左侧 ▶ 即可）
#@markdown 点一下左边的 `▶`，会自动完成：**挂载云盘 $\rightarrow$ 安装 ComfyUI $\rightarrow$ 生成专属网页链接**。\
#@markdown 等约 2 分钟，下方出现 `🚀 【点击进入 ComfyUI】` 链接时，直接点它就能画图！

import os, subprocess, threading, re
from google.colab import drive

print("🔍 当前分配显卡:", subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))
drive.mount('/content/drive')

STUDIO_ROOT = "/content/drive/MyDrive/AI_Manga_Studio"
for sub in ["datasets", "loras", "checkpoints", "output", "configs"]:
    os.makedirs(f"{STUDIO_ROOT}/{sub}", exist_ok=True)

if not os.path.exists("/content/ComfyUI"):
    print("⚙️ 正在安装 ComfyUI 及插件管理器（约需 2 分钟）...")
    !git clone -q https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !pip install -q -r /content/ComfyUI/requirements.txt

!rm -rf /content/ComfyUI/models/loras /content/ComfyUI/models/checkpoints /content/ComfyUI/output
!ln -s "{STUDIO_ROOT}/loras" /content/ComfyUI/models/loras
!ln -s "{STUDIO_ROOT}/checkpoints" /content/ComfyUI/models/checkpoints
!ln -s "{STUDIO_ROOT}/output" /content/ComfyUI/output

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

def start_tunnel(port):
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in p.stderr:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            print(f"\n{'='*65}")
            print(f"🚀 【点击进入 ComfyUI 漫画工作台】: {m.group(0)}")
            print(f"{'='*65}\n")

threading.Thread(target=start_tunnel, args=(8188,), daemon=True).start()
!python /content/ComfyUI/main.py --dont-print-server

In [ ]:
#@title ## 🔥 【按钮 2】一键全自动训练 Krea-2-Raw 专属画风/角色 LoRA（点左侧 ▶ 即可）
#@markdown 当我在你的 Mac 上帮你扒好 Danbooru 原画、打好自然语言标签并同步后，你只需要点一下左边的 `▶`！\
#@markdown 它会自动拉取训练集、根据 A100/L4 自动开启最优配置，并将练好的 LoRA 实时存进你的 Google 云端硬盘。

DATASET_NAME = "inoue_style" #@param {type:"string"}
LORA_NAME = "krea2_inoue_manga_v1" #@param {type:"string"}
TRIGGER_WORD = "by_inoue_style" #@param {type:"string"}
STEPS = 2000 #@param {type:"integer"}
LEARNING_RATE = "1e-4" #@param ["1e-4", "8e-5", "5e-5"]
LORA_RANK = 32 #@param [16, 32, 64]

import os, subprocess, zipfile, yaml
from google.colab import drive

gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"🔍 当前显卡: {gpu_info}")
drive.mount('/content/drive')

STUDIO_ROOT = "/content/drive/MyDrive/AI_Manga_Studio"
for sub in ["datasets", "loras", "checkpoints", "output", "configs"]:
    os.makedirs(f"{STUDIO_ROOT}/{sub}", exist_ok=True)

# 自动从 GitHub 同步最新打包的数据集（免去手动上传云盘！）
if not os.path.exists("/content/krea2-manga-studio"):
    !git clone -q https://github.com/zsdlz971004-boop/krea2-manga-studio.git /content/krea2-manga-studio
else:
    !cd /content/krea2-manga-studio && git pull -q

extract_dir = f"/content/dataset/{DATASET_NAME}"
os.makedirs(extract_dir, exist_ok=True)

repo_zip = f"/content/krea2-manga-studio/datasets/{DATASET_NAME}.zip"
drive_zip = f"{STUDIO_ROOT}/datasets/{DATASET_NAME}.zip"
target_zip = repo_zip if os.path.exists(repo_zip) else (drive_zip if os.path.exists(drive_zip) else None)

if target_zip:
    print(f"📦 自动加载训练集: {target_zip}")
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall(extract_dir)
    print(f"✅ 训练集就绪，共 {len(os.listdir(extract_dir))} 个文件！")
else:
    raise FileNotFoundError(f"未找到训练集 {DATASET_NAME}.zip！请先在聊天窗口告诉 Antigravity 你想练哪位画师，我会自动帮你扒图打标并同步过来。")

if not os.path.exists("/content/ai-toolkit"):
    print("⚙️ 正在安装 ostris/ai-toolkit 训练引擎...")
    !git clone -q https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
    !cd /content/ai-toolkit && git submodule update --init --recursive
    !pip install -q -r /content/ai-toolkit/requirements.txt

is_a100 = "A100" in gpu_info or "80" in gpu_info or "40960" in gpu_info
quantize = not is_a100
low_vram = not is_a100
print("🚀 模式:", "A100 BF16 满血极速训练" if is_a100 else "L4 24GB FP8 防爆显存安全训练")

config = {
    "job": "extension",
    "config": {
        "name": LORA_NAME,
        "process": [{
            "type": "sd_trainer",
            "training_folder": f"{STUDIO_ROOT}/loras",
            "device": "cuda:0",
            "trigger_word": TRIGGER_WORD,
            "network": {"type": "lora", "linear": int(LORA_RANK), "linear_alpha": int(LORA_RANK)},
            "save": {"dtype": "float16", "save_every": 500, "max_step_saves_to_keep": 4},
            "datasets": [{
                "folder_path": extract_dir,
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "cache_text_embeddings": True,
                "resolution": [1024]
            }],
            "train": {
                "batch_size": 1,
                "steps": int(STEPS),
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "optimizer": "adamw8bit",
                "lr": float(LEARNING_RATE),
                "lr_scheduler": "cosine",
                "dtype": "bf16"
            },
            "model": {
                "name_or_path": "krea/Krea-2-Raw",
                "arch": "krea_2",
                "quantize": quantize,
                "low_vram": low_vram
            }
        }]
    }
}

cfg_path = f"{STUDIO_ROOT}/configs/{LORA_NAME}.yaml"
with open(cfg_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f, sort_keys=False, allow_unicode=True)

print(f"🔥 开始全自动训练！权重将自动保存在云盘: {STUDIO_ROOT}/loras/")
!python /content/ai-toolkit/run.py "{cfg_path}"